In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.covariance import LedoitWolf
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

CACHE_DIR  = Path("../embeddings/cache")
OUTPUT_DIR = Path("../data/subsets")

In [2]:
# Cell 2 — Load embeddings and index
embeddings = np.load(CACHE_DIR / "embeddings_resnet18.npy")
index_df   = pd.read_csv(CACHE_DIR / "embedding_index.csv")

print("Embeddings shape:", embeddings.shape)
print("Index shape:     ", index_df.shape)
print(index_df["dataset"].value_counts())
print(index_df["label"].value_counts())

Embeddings shape: (17934, 512)
Index shape:      (17934, 7)
dataset
isic        9158
chexpert    8776
Name: count, dtype: int64
label
BCC                           2500
MEL                           2292
NV                            2235
BKL                           2131
No Finding                    1464
Consolidation                  806
Pneumonia                      743
Edema                          608
Enlarged Cardiomediastinum     580
Pneumothorax                   533
Fracture                       533
Support Devices                532
Cardiomegaly                   530
Pleural Effusion               504
Atelectasis                    502
Lung Lesion                    482
Lung Opacity                   480
Pleural Other                  479
Name: count, dtype: int64


In [3]:
# Cell 3 — Verify alignment
assert len(embeddings) == len(index_df), "Mismatch between embeddings and index"
assert (index_df["embedding_row"] == range(len(index_df))).all(), "Row order mismatch"
print("Alignment verified.")

Alignment verified.


In [ ]:
# Cell 4 — kNN Scorer (per dataset)
# Nearest neighbours computed ONLY within each separate dataset's own embedding vector space
# This prevents CheXpert images from being neighbours of ISIC images to avoid computing on same metrics

K = 15
print(f"Running kNN scorer (k={K}) separately per dataset...\n")

knn_scores = np.zeros(len(embeddings))

for dataset_name in ["chexpert", "isic"]:
    ds_mask    = index_df["dataset"].values == dataset_name
    ds_indices = np.where(ds_mask)[0]
    ds_embeddings = embeddings[ds_mask]
    ds_labels     = index_df["label"].values[ds_mask]

    print(f"  {dataset_name}: {len(ds_embeddings)} images")

    knn = NearestNeighbors(n_neighbors=K+1, metric="euclidean", n_jobs=-1)
    knn.fit(ds_embeddings)
    distances, indices = knn.kneighbors(ds_embeddings)

    # indices[:,0] is the image itself — skip it
    neighbour_indices = indices[:, 1:]

    for local_i, global_i in enumerate(ds_indices):
        own_label = ds_labels[local_i]
        neighbour_labels = ds_labels[neighbour_indices[local_i]]
        different = np.sum(neighbour_labels != own_label)
        knn_scores[global_i] = different / K

index_df["knn_score"] = knn_scores
print("\nkNN scoring done.")
print(f"CheXpert — mean kNN score: {index_df[index_df['dataset']=='chexpert']['knn_score'].mean():.3f}")
print(f"ISIC     — mean kNN score: {index_df[index_df['dataset']=='isic']['knn_score'].mean():.3f}")

Running kNN scorer (k=15) separately per dataset...

  chexpert: 8776 images
  isic: 9158 images

kNN scoring done.
CheXpert — mean kNN score: 0.880
ISIC     — mean kNN score: 0.495


In [5]:
# Diagnostic
print("Embedding index columns:", index_df.columns.tolist())
print("Dataset values:", index_df["dataset"].unique())
print("Dataset counts:", index_df["dataset"].value_counts())
print("\nSample rows:")
print(index_df.head(3))

Embedding index columns: ['image_id', 'dataset', 'label', 'is_uncertain_original', 'kaggle_path', 'embedding_row', 'extraction_failed', 'knn_score']
Dataset values: <StringArray>
['chexpert', 'isic']
Length: 2, dtype: str
Dataset counts: dataset
isic        9158
chexpert    8776
Name: count, dtype: int64

Sample rows:
            image_id   dataset       label  is_uncertain_original  \
0  view1_frontal.jpg  chexpert  No Finding                  False   
1  view1_frontal.jpg  chexpert  No Finding                  False   
2  view1_frontal.jpg  chexpert  No Finding                  False   

                                         kaggle_path  embedding_row  \
0  /kaggle/input/datasets/ashery/chexpert/train/p...              0   
1  /kaggle/input/datasets/ashery/chexpert/train/p...              1   
2  /kaggle/input/datasets/ashery/chexpert/train/p...              2   

   extraction_failed  knn_score  
0              False   0.866667  
1              False   0.933333  
2              F

In [ ]:
# Cell 5 — Mahalanobis Distance Scorer (per dataset, per class)
# This measures how far each image sits from its own class centroid
# LedoitWolf handles high-dimensional covariance estimation reliably

print("Computing Mahalanobis distances per dataset per class...\n")

mahal_scores = np.zeros(len(embeddings))

for dataset_name in ["chexpert", "isic"]:
    ds_mask    = index_df["dataset"].values == dataset_name
    ds_indices = np.where(ds_mask)[0]
    ds_labels  = index_df["label"].values[ds_mask]
    ds_classes = np.unique(ds_labels)

    print(f"  {dataset_name}: {len(ds_classes)} classes")

    for cls in tqdm(ds_classes, desc=f"  {dataset_name} Mahalanobis"):
        cls_local_mask  = ds_labels == cls
        cls_global_idx  = ds_indices[cls_local_mask]
        cls_embeddings  = embeddings[cls_global_idx]

        if len(cls_embeddings) < 5:
            mahal_scores[cls_global_idx] = 0.0
            continue

        lw = LedoitWolf()
        lw.fit(cls_embeddings)
        centroid = cls_embeddings.mean(axis=0)
        cov_inv  = np.linalg.inv(lw.covariance_)

        for local_i, global_i in enumerate(cls_global_idx):
            diff  = embeddings[global_i] - centroid
            score = np.sqrt(diff @ cov_inv @ diff)
            mahal_scores[global_i] = score

index_df["mahal_score"] = mahal_scores
print("\nMahalanobis scoring done.")
print(f"CheXpert — mean: {index_df[index_df['dataset']=='chexpert']['mahal_score'].mean():.3f}")
print(f"ISIC     — mean: {index_df[index_df['dataset']=='isic']['mahal_score'].mean():.3f}")

Computing Mahalanobis distances per dataset per class...

  chexpert: 14 classes


  chexpert Mahalanobis: 100%|██████████| 14/14 [00:03<00:00,  3.70it/s]


  isic: 4 classes


  isic Mahalanobis: 100%|██████████| 4/4 [00:01<00:00,  3.75it/s]


Mahalanobis scoring done.
CheXpert — mean: 17.682
ISIC     — mean: 20.767


In [ ]:
# Cell 6 — Isolation Forest Scorer (per dataset, per class)
# Trained independently within each dataset's class 
# Detects images statistically unusual within their own classes

print("Running Isolation Forest per dataset per class...\n")

iso_scores = np.zeros(len(embeddings))

for dataset_name in ["chexpert", "isic"]:
    ds_mask    = index_df["dataset"].values == dataset_name
    ds_indices = np.where(ds_mask)[0]
    ds_labels  = index_df["label"].values[ds_mask]
    ds_classes = np.unique(ds_labels)

    print(f"  {dataset_name}: {len(ds_classes)} classes")

    for cls in tqdm(ds_classes, desc=f"  {dataset_name} IsoForest"):
        cls_local_mask = ds_labels == cls
        cls_global_idx = ds_indices[cls_local_mask]
        cls_embeddings = embeddings[cls_global_idx]

        if len(cls_embeddings) < 10:
            iso_scores[cls_global_idx] = 0.0
            continue

        iso = IsolationForest(
            n_estimators=100,
            contamination=0.1,
            random_state=42,
            n_jobs=-1
        )
        iso.fit(cls_embeddings)

        # Negate so higher = more suspicious
        raw = -iso.decision_function(cls_embeddings)
        iso_scores[cls_global_idx] = raw

index_df["iso_score"] = iso_scores
print("\nIsolation Forest scoring done.")
print(f"CheXpert — mean: {index_df[index_df['dataset']=='chexpert']['iso_score'].mean():.3f}")
print(f"ISIC     — mean: {index_df[index_df['dataset']=='isic']['iso_score'].mean():.3f}")

Running Isolation Forest per dataset per class...

  chexpert: 14 classes


  chexpert IsoForest: 100%|██████████| 14/14 [00:01<00:00,  8.15it/s]


  isic: 4 classes


  isic IsoForest: 100%|██████████| 4/4 [00:00<00:00,  6.58it/s]


Isolation Forest scoring done.
CheXpert — mean: -0.032
ISIC     — mean: -0.046


In [ ]:
# Cell 7 — Rank Fusion (per dataset independently)
# Ranks computed within each dataset so scores are comparable 
# caliculated within domain — not mixed across chest X-rays and skin lesions datasets

print("Fusing scores via rank averaging (per dataset)...\n")

index_df["knn_rank"]   = 0.0
index_df["mahal_rank"] = 0.0
index_df["iso_rank"]   = 0.0

for dataset_name in ["chexpert", "isic"]:
    ds_mask = index_df["dataset"] == dataset_name

    index_df.loc[ds_mask, "knn_rank"] = (
        index_df.loc[ds_mask, "knn_score"].rank(ascending=False)
    )
    index_df.loc[ds_mask, "mahal_rank"] = (
        index_df.loc[ds_mask, "mahal_score"].rank(ascending=False)
    )
    index_df.loc[ds_mask, "iso_rank"] = (
        index_df.loc[ds_mask, "iso_score"].rank(ascending=False)
    )

index_df["composite_rank"] = (
    index_df["knn_rank"] +
    index_df["mahal_rank"] +
    index_df["iso_rank"]
) / 3

# Normalise suspicion score within each dataset (0 to 1)
for dataset_name in ["chexpert", "isic"]:
    ds_mask  = index_df["dataset"] == dataset_name
    max_rank = ds_mask.sum()
    index_df.loc[ds_mask, "suspicion_score"] = (
        index_df.loc[ds_mask, "composite_rank"] / max_rank
    )

index_df = index_df.sort_values(
    ["dataset", "suspicion_score"], ascending=[True, False]
).reset_index(drop=True)

print("Rank fusion done.\n")
print("Top 5 most suspicious — CheXpert:")
chex = index_df[index_df["dataset"] == "chexpert"]
print(chex[["image_id", "label", "knn_score", 
            "mahal_score", "suspicion_score"]].head(5).to_string())

print("\nTop 5 most suspicious — ISIC:")
isic = index_df[index_df["dataset"] == "isic"]
print(isic[["image_id", "label", "knn_score",
            "mahal_score", "suspicion_score"]].head(5).to_string())

Fusing scores via rank averaging (per dataset)...

Rank fusion done.

Top 5 most suspicious — CheXpert:
            image_id          label  knn_score  mahal_score  suspicion_score
0  view1_frontal.jpg   Lung Opacity   0.733333    14.217340         0.961581
1  view1_frontal.jpg  Pleural Other   0.733333    13.562653         0.957289
2  view1_frontal.jpg       Fracture   0.733333    14.196107         0.953832
3  view1_frontal.jpg    Atelectasis   0.666667    15.466106         0.948515
4  view1_frontal.jpg   Pneumothorax   0.733333    14.738503         0.944793

Top 5 most suspicious — ISIC:
          image_id label  knn_score  mahal_score  suspicion_score
8776  ISIC_0029887    NV        0.0    10.856149         0.993266
8777  ISIC_0030393    NV        0.0    11.550565         0.991046
8778  ISIC_0024419    NV        0.0    12.617391         0.989372
8779  ISIC_0029739    NV        0.0    12.436890         0.989335
8780  ISIC_0028491    NV        0.0    12.615149         0.989263


In [9]:
# Cell 8 — Weak Domain Rules (per dataset)
print("Applying weak domain rules...\n")

# Rule 1 — CheXpert only: originally uncertain labels
index_df["rule_uncertain_label"] = (
    (index_df["dataset"] == "chexpert") &
    (index_df["is_uncertain_original"] == True)
)

# Rule 2 — High kNN score (within-dataset threshold)
index_df["rule_high_knn"] = False
for dataset_name in ["chexpert", "isic"]:
    ds_mask   = index_df["dataset"] == dataset_name
    threshold = index_df.loc[ds_mask, "knn_score"].quantile(0.85)
    index_df.loc[ds_mask, "rule_high_knn"] = (
        index_df.loc[ds_mask, "knn_score"] >= threshold
    )

# Rule 3 — Extreme Mahalanobis (top 5% within each dataset)
index_df["rule_extreme_mahal"] = False
for dataset_name in ["chexpert", "isic"]:
    ds_mask   = index_df["dataset"] == dataset_name
    threshold = index_df.loc[ds_mask, "mahal_score"].quantile(0.95)
    index_df.loc[ds_mask, "rule_extreme_mahal"] = (
        index_df.loc[ds_mask, "mahal_score"] >= threshold
    )

# Rule 4 — All three scorers agree (top 20% in each, within dataset)
index_df["rule_all_three_agree"] = False
for dataset_name in ["chexpert", "isic"]:
    ds_mask = index_df["dataset"] == dataset_name
    t_knn   = index_df.loc[ds_mask, "knn_score"].quantile(0.80)
    t_mahal = index_df.loc[ds_mask, "mahal_score"].quantile(0.80)
    t_iso   = index_df.loc[ds_mask, "iso_score"].quantile(0.80)

    index_df.loc[ds_mask, "rule_all_three_agree"] = (
        (index_df.loc[ds_mask, "knn_score"]   >= t_knn)   &
        (index_df.loc[ds_mask, "mahal_score"] >= t_mahal) &
        (index_df.loc[ds_mask, "iso_score"]   >= t_iso)
    )

rule_cols = ["rule_uncertain_label", "rule_high_knn",
             "rule_extreme_mahal", "rule_all_three_agree"]
index_df["rules_fired"] = index_df[rule_cols].sum(axis=1)

print("Weak rules applied.")
print("\nRule trigger counts:")
for col in rule_cols:
    chex_count = index_df[index_df["dataset"]=="chexpert"][col].sum()
    isic_count = index_df[index_df["dataset"]=="isic"][col].sum()
    print(f"  {col}: CheXpert={chex_count}  ISIC={isic_count}")

Applying weak domain rules...

Weak rules applied.

Rule trigger counts:
  rule_uncertain_label: CheXpert=2573  ISIC=0
  rule_high_knn: CheXpert=2273  ISIC=1554
  rule_extreme_mahal: CheXpert=439  ISIC=458
  rule_all_three_agree: CheXpert=171  ISIC=416


In [10]:
# Cell 9 — Per-class suspicion summary
print("Per-class suspicion summary:\n")

summary = index_df.groupby(["dataset", "label"]).agg(
    total_images   = ("image_id", "count"),
    mean_suspicion = ("suspicion_score", "mean"),
    high_suspicion = ("suspicion_score", lambda x: (x > 0.7).sum()),
    rules_fired    = ("rules_fired", "sum")
).sort_values(["dataset", "mean_suspicion"], ascending=[True, False])

print(summary.to_string())

Per-class suspicion summary:

                                     total_images  mean_suspicion  high_suspicion  rules_fired
dataset  label                                                                                
chexpert Atelectasis                          502        0.559545             149          225
         Pleural Other                        479        0.538488             101          368
         Lung Lesion                          482        0.537763             109          238
         Lung Opacity                         480        0.530989              89          223
         Support Devices                      532        0.528997             116          175
         Cardiomegaly                         530        0.522254              88          296
         Pneumothorax                         533        0.519962              81          289
         Consolidation                        806        0.493672             145          852
         Pleural Eff

In [11]:
# Cell 10 — Save audit scores
output_path = OUTPUT_DIR / "label_audit_scores.csv"
index_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Total images audited: {len(index_df)}")
print(f"Columns: {index_df.columns.tolist()}")

Saved: ..\data\subsets\label_audit_scores.csv
Total images audited: 17934
Columns: ['image_id', 'dataset', 'label', 'is_uncertain_original', 'kaggle_path', 'embedding_row', 'extraction_failed', 'knn_score', 'mahal_score', 'iso_score', 'knn_rank', 'mahal_rank', 'iso_rank', 'composite_rank', 'suspicion_score', 'rule_uncertain_label', 'rule_high_knn', 'rule_extreme_mahal', 'rule_all_three_agree', 'rules_fired']


In [12]:
# Cell 11 — Final sanity check
audit = pd.read_csv(OUTPUT_DIR / "label_audit_scores.csv")
assert len(audit) == len(index_df), "Row count mismatch"

print("File verified.\n")

for dataset_name in ["chexpert", "isic"]:
    ds = audit[audit["dataset"] == dataset_name]
    print(f"{dataset_name.upper()} — {len(ds)} images")
    print(f"  Suspicion score range: "
          f"{ds['suspicion_score'].min():.3f} to "
          f"{ds['suspicion_score'].max():.3f}")
    print(f"  High suspicion (>0.7): {(ds['suspicion_score'] > 0.7).sum()}")
    print(f"  All-three-agree flags: {ds['rule_all_three_agree'].sum()}")
    print()

File verified.

CHEXPERT — 8776 images
  Suspicion score range: 0.046 to 0.962
  High suspicion (>0.7): 1355
  All-three-agree flags: 171

ISIC — 9158 images
  Suspicion score range: 0.003 to 0.993
  High suspicion (>0.7): 1900
  All-three-agree flags: 416

